# Customer Retention & RFM Analysis — RFM Segmentation

This notebook takes the Recency, Frequency, and Monetary values calculated in SQL and scores each customer into segments (Champions, Loyal, At Risk, Lost, etc.) to identify who the business should we prioritize retaining.

## 1. Load RFM Ingredients

In [5]:
import pandas as pd

rfm = pd.read_csv('/Users/piyushkalra/Desktop/rfm_raw.csv')
print(rfm.head())
print(rfm.describe())
rfm.info()

   Customer ID  recency  frequency  monetary
0        12346      164         11    372.86
1        12347        2          2   1323.32
2        12348       73          1    222.16
3        12349       42          3   2671.14
4        12351       10          1    300.93
        Customer ID      recency    frequency       monetary
count   4300.000000  4300.000000  4300.000000    4300.000000
mean   15350.424651    90.159302     4.441860    2023.186454
std     1702.032144    96.992586     8.024423    8828.596190
min    12346.000000     0.000000     1.000000       0.000000
25%    13882.500000    17.000000     1.000000     306.525000
50%    15352.500000    51.000000     2.000000     700.405000
75%    16835.250000   135.000000     5.000000    1705.540000
max    18287.000000   373.000000   188.000000  349164.350000
<class 'pandas.DataFrame'>
RangeIndex: 4300 entries, 0 to 4299
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0

## 2. Score Each Customer on R, F, M (1–5 scale)

Divided customers into 5 equal-sized groups on each dimension. For Recency, a LOWER number of days is better (more recent), so the scoring is reversed compared to Frequency and Monetary, where HIGHER is better.

In [6]:
rfm['R_score'] = pd.qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M_score'] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5])

rfm.head(10)

,Customer ID,recency,frequency,monetary,R_score,F_score,M_score
0,12346,164,11,372.86,2,5,2
1,12347,2,2,1323.32,5,2,4
2,12348,73,1,222.16,2,1,1
3,12349,42,3,2671.14,3,3,5
4,12351,10,1,300.93,5,1,2
5,12352,10,2,343.80,5,2,2
6,12353,43,1,317.76,3,1,2
7,12355,202,1,488.21,1,1,2
8,12356,15,3,3560.30,4,3,5
9,12357,23,2,12079.99,4,2,5


## 3. Validation: Cross-Check Against SQL Findings

Before trusting the RFM scores, I checked that the top 10 customers by revenue identified directly in SQL also score highly on RFM, this confirms the scoring logic is working correctly rather than producing arbitrary results.

In [7]:
top_10_ids = ['18102','14646','14156','14911','13694','17511','15061','16684','16754','13089']

champions_check = rfm[rfm['Customer ID'].astype(str).isin(top_10_ids)][['Customer ID', 'R_score', 'F_score', 'M_score']]
champions_check

,Customer ID,R_score,F_score,M_score
505,13089,5,5,5
937,13694,5,5,5
1267,14156,5,5,5
1632,14646,5,5,5
1833,14911,5,5,5
1944,15061,5,5,5
3118,16684,5,5,5
3167,16754,5,5,5
3733,17511,5,5,5
4171,18102,5,5,5


Validation result: All top 10 customers identified independently via SQL scored 5 (or near-5) across Recency, Frequency, and Monetary confirming the RFM scoring logic correctly identifies high-value customers, consistent with the raw revenue based query.

## 4. Assign Customer Segments

Combined R and F scores into human-readable segments a business can act on.

In [8]:
def segment_customer(row):
    r = int(row['R_score'])
    f = int(row['F_score'])
    
    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2:
        return 'Lost'
    else:
        return 'Potential'

rfm['segment'] = rfm.apply(segment_customer, axis=1)

rfm['segment'].value_counts()

segment
Champions    1105
Lost         1022
Loyal         778
Potential     698
At Risk       697
Name: count, dtype: int64

Finding: Of ~4,300 customers, 1,105 (26%) are Champions- the most valuable, recently-active, frequent buyers. 697 customers (16%) are At Risk — customers who used to engage regularly but have gone quiet, representing the core group for retention efforts.

## 5. Revenue by Segment

Quantifying how much total monetary value each segment represents, this turns segment counts into an actual business number.

In [12]:
segment_revenue = rfm.groupby('segment')['monetary'].agg(['sum', 'mean', 'count']).sort_values('sum', ascending=False)
segment_revenue.columns = ['total_revenue', 'avg_revenue_per_customer', 'customer_count']
print(segment_revenue)


           total_revenue  avg_revenue_per_customer  customer_count
segment                                                           
Champions    5804350.508               5252.805890            1105
Loyal        1145016.011               1471.742945             778
At Risk      1025870.131               1471.836630             697
Lost          396594.504                388.057245            1022
Potential     327870.600                469.728653             698


In [13]:
#finding the % of revenue generated by the at-risk customers
total_revenue = segment_revenue['total_revenue'].sum()
at_risk_pct = (segment_revenue.loc['At Risk', 'total_revenue'] / total_revenue) * 100
print(f"At-Risk revenue: £{segment_revenue.loc['At Risk', 'total_revenue']:,.0f} ({at_risk_pct:.1f}% of total)")

At-Risk revenue: £1,025,870 (11.8% of total)


Key finding: At-Risk customers represent ~£1.03M (11.8%) of total revenue and their average value (£1,471.84) is nearly identical to the Loyal segment (£1,471.74). This means At-Risk customers aren't inherently lower-value, they were just as valuable as currently loyal customers before going quiet, making them a strong target for retention efforts rather than a lost cause.

## 6. ROI of a Win-Back Campaign

Designing a hypothetical discount campaign targeted at At-Risk customers, and comparing a targeted approach (top-value At-Risk customers only) against a blanket approach (all At-Risk customers) to see which delivers better return on investment.

In [14]:
# Assumptions- stated explicitly so they can be challenged/adjusted
discount_pct = 0.10
reactivation_rate_targeted = 0.15
reactivation_rate_blanket = 0.12
campaign_cost_per_customer = 5  # cost to send a discount email/SMS

at_risk = rfm[rfm['segment'] == 'At Risk']
n_at_risk = len(at_risk)
avg_monetary_at_risk = at_risk['monetary'].mean()

print(f"At-Risk customers: {n_at_risk}")
print(f"Average value per At-Risk customer: £{avg_monetary_at_risk:,.2f}")

At-Risk customers: 697
Average value per At-Risk customer: £1,471.84


In [ ]:
# Targeted campaign- top 30% of At-Risk customers by value
top_30_pct_count = int(n_at_risk * 0.3)
top_targeted = at_risk.nlargest(top_30_pct_count, 'monetary')

recovered_revenue_targeted = top_targeted['monetary'].sum() * reactivation_rate_targeted * (1 - discount_pct)
cost_targeted = top_30_pct_count * campaign_cost_per_customer
roi_targeted = (recovered_revenue_targeted - cost_targeted) / cost_targeted

print(f"Targeted campaign — customers reached: {top_30_pct_count}")
print(f"Recovered revenue: £{recovered_revenue_targeted:,.2f}")
print(f"Campaign cost: £{cost_targeted:,.2f}")
print(f"ROI: {roi_targeted:.1f}x")

Targeted campaign — customers reached: 209
Recovered revenue: £95,039.05
Campaign cost: £1,045.00
ROI: 89.9x


In [16]:
# Blanket campaign- all At-Risk customers
recovered_revenue_blanket = at_risk['monetary'].sum() * reactivation_rate_blanket * (1 - discount_pct)
cost_blanket = n_at_risk * campaign_cost_per_customer
roi_blanket = (recovered_revenue_blanket - cost_blanket) / cost_blanket

print(f"Blanket campaign — customers reached: {n_at_risk}")
print(f"Recovered revenue: £{recovered_revenue_blanket:,.2f}")
print(f"Campaign cost: £{cost_blanket:,.2f}")
print(f"ROI: {roi_blanket:.1f}x")

Blanket campaign — customers reached: 697
Recovered revenue: £110,793.97
Campaign cost: £3,485.00
ROI: 30.8x


## 7. Sensitivity Analysis

I Tested the ROI finding across a range of realistic assumptions, rather than relying on a single best case scenario,this checks whether the conclusion (targeted beats blanket) holds up even under more conservative estimates.

In [17]:
print(f"{'Cost/Customer':<15}{'Reactivation':<15}{'ROI (Targeted)':<18}{'ROI (Blanket)'}")

for cost in [5, 10, 20]:
    for reactivation in [0.10, 0.15, 0.20]:
        # Targeted
        recovered_t = top_targeted['monetary'].sum() * reactivation * (1 - discount_pct)
        cost_t = top_30_pct_count * cost
        roi_t = (recovered_t - cost_t) / cost_t
        
        # Blanket
        recovered_b = at_risk['monetary'].sum() * reactivation * (1 - discount_pct)
        cost_b = n_at_risk * cost
        roi_b = (recovered_b - cost_b) / cost_b
        
        print(f"£{cost:<14}{reactivation*100:.0f}%{'':<12}{roi_t:.1f}x{'':<13}{roi_b:.1f}x")

Cost/Customer  Reactivation   ROI (Targeted)    ROI (Blanket)
£5             10%            59.6x             25.5x
£5             15%            89.9x             38.7x
£5             20%            120.3x             52.0x
£10            10%            29.3x             12.2x
£10            15%            44.5x             18.9x
£10            20%            59.6x             25.5x
£20            10%            14.2x             5.6x
£20            15%            21.7x             8.9x
£20            20%            29.3x             12.2x


## Summary

- Segmented ~4,300 customers into 5 groups using RFM (Recency, Frequency, Monetary) scoring, validated against independent SQL findings.
- Identified ~£1.03M (11.8%) of total revenue concentrated in At-Risk customers comparable in average value to Loyal customers, meaning these are high-value customers at genuine risk of churning, not low-value ones drifting away naturally.
- Modeled a win-back campaign and found that targeting the top 30% of At-Risk customers by value delivers roughly 2.4-2.5x better ROI than a blanket campaign a conclusion that holds consistently across 9 different cost/reactivation assumption combinations.

Next steps: I will now build a Power BI dashboard to make these findings actionable for a business user, including a ranked "contact this week" list of At-Risk customers by value.

In [ ]:
#saving the file to move to Power Bi phase
rfm.to_csv('/Users/piyushkalra/Desktop/rfm_final.csv', index=False)
print("Final RFM segments saved for Power BI dashboard")

Final RFM segments saved for Power BI dashboard
